# MCMC — distribuições de erro dos surrogates

- Lê `df_{nome}_previsao.parquet`.
- Amostra de validação estratificada por região.
- KS + figuras + MCMC; grava `df_{nome}_mcmc.parquet`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

from src.statistics import comparar_distribuicoes, gerar_amostras_mcmc_por_problema
from src.visualization import plota_comparacao_distribuicoes

PROBLEM_NAMES = [
    'MMF1', 'MMF4', 'MMF11_L', 'MMF16_L3', 'MMF16_20',
    'ZDT1', 'ZDT3', 'ZDT4', 'ZDT6',
    'DTLZ1', 'DTLZ2', 'DTLZ3', 'DTLZ4', 'DTLZ7',
    'WFG1', 'WFG2', 'WFG4', 'WFG5', 'WFG9',
]

THREE_OBJ = {
    'MMF16_L3', 'MMF16_20', 'DTLZ1', 'DTLZ2', 'DTLZ3', 'DTLZ4', 'DTLZ7',
}


def mcmc_pipeline(
    name,
    obs_validacao=50_000,
    n_samples_mcmc=50,
    run_plots=True,
):
    """Carrega previsões, erros, amostra validação, KS, MCMC, gráficos opcionais, salva parquet MCMC."""

    # Carrega previsões
    base = Path('data/dataframes') / name
    path_pre = base / f'df_{name}_previsao.parquet'
    df = pd.read_parquet(path_pre)

    # Calcula erros
    df['erro_f1'] = df['f1'] - df['f1_predicted']
    df['erro_f2'] = df['f2'] - df['f2_predicted']
    err_cols = ['erro_f1', 'erro_f2']
    if name in THREE_OBJ:
        df['erro_f3'] = df['f3'] - df['f3_predicted']
        err_cols.append('erro_f3')

    # Coleta amostra de dados de validação estratificada por região
    n_regioes = 16
    n_per = int(obs_validacao / n_regioes)

    def _sample_grp(g):
        k = min(n_per, len(g))
        return g.sample(n=k, random_state=42)

    df_val = df.groupby('regiao', group_keys=False).apply(_sample_grp)

    # Valida dados de validação (análise de distribuições vs. geral: KS test e figuras)
    dfs_ks = [comparar_distribuicoes(df, df_val, col=c, problema=name) for c in err_cols]
    df_ks = pd.concat(dfs_ks, ignore_index=True)
    display(df_ks.head(8))

    if run_plots:
        fig = plota_comparacao_distribuicoes(
            df, df_val, df_ks,
            problem_name=name,
            max_cols=4,
            fitness_cols=err_cols,
            fitness_labels=[c.replace('erro_', '').upper() for c in err_cols],
        )
        plt.show()

    # Gera amostras MCMC por região a partir dos dados de validação
    col3 = 'erro_f3' if name in THREE_OBJ else None
    df_mcmc = gerar_amostras_mcmc_por_problema(
        df_validacao=df_val,
        n_samples=n_samples_mcmc,
        col1='erro_f1',
        col2='erro_f2',
        col3=col3,
    )

    out_mcmc = base / f'df_{name}_mcmc.parquet'
    df_mcmc.to_parquet(out_mcmc)
    print('Salvo', out_mcmc, df_mcmc.shape)

    # Valida distribuições MCMC vs. validação
    if run_plots:
        dfs_mcmc = [
            comparar_distribuicoes(df_mcmc, df_val, col=c, problema=name, calcula_wape=False)
            for c in err_cols
        ]
        df_ks_mcmc = pd.concat(dfs_mcmc, ignore_index=True)
        plota_comparacao_distribuicoes(
            df_mcmc, df_val, df_ks_mcmc,
            problem_name=name,
            max_cols=4,
            fitness_cols=err_cols,
            fitness_labels=[c.replace('erro_', '').upper() for c in err_cols],
        )
        plt.show()

    return {'df': df, 'df_validacao': df_val, 'df_ks': df_ks, 'df_mcmc': df_mcmc}


## MMF1

In [ ]:
mcmc_pipeline('MMF1')


## MMF4

In [ ]:
mcmc_pipeline('MMF4')


## MMF11_L

In [ ]:
mcmc_pipeline('MMF11_L')


## MMF16_L3

In [ ]:
mcmc_pipeline('MMF16_L3')


## MMF16_20

In [ ]:
mcmc_pipeline('MMF16_20')


## ZDT1

In [ ]:
mcmc_pipeline('ZDT1')


## ZDT3

In [ ]:
mcmc_pipeline('ZDT3')


## ZDT4

In [ ]:
mcmc_pipeline('ZDT4')


## ZDT6

In [ ]:
mcmc_pipeline('ZDT6')


## DTLZ1

In [ ]:
mcmc_pipeline('DTLZ1')


## DTLZ2

In [ ]:
mcmc_pipeline('DTLZ2')


## DTLZ3

In [ ]:
mcmc_pipeline('DTLZ3')


## DTLZ4

In [ ]:
mcmc_pipeline('DTLZ4')


## DTLZ7

In [ ]:
mcmc_pipeline('DTLZ7')


## WFG1

In [ ]:
mcmc_pipeline('WFG1')


## WFG2

In [ ]:
mcmc_pipeline('WFG2')

## WFG4

In [ ]:
mcmc_pipeline('WFG4')


## WFG5

In [ ]:
mcmc_pipeline('WFG5')


## WFG9

In [ ]:
mcmc_pipeline('WFG9')
